<a href="https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UbaidMalik365/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [11]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

# Features used for the content-refresh analysis
numeric_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "char_count"
]

categorical_features = [
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

# Keep only features that actually exist
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

# Numeric feature matrix
X_numeric = df[numeric_features].copy()

# Fill missing numeric values with the median
X_numeric = X_numeric.fillna(X_numeric.median(numeric_only=True))

# One-hot encode categorical variables
X_categorical = pd.get_dummies(
    df[categorical_features],
    dummy_na=True,
    dtype=int
)

# Combine numeric and categorical features
X = pd.concat([X_numeric, X_categorical], axis=1)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Final feature matrix shape:", X.shape)

display(X.head())

Dataset shape: (30000, 44)
Number of columns: 44
Numeric features: 19
Categorical features: 8
Final feature matrix shape: (30000, 59)


,content_age_days,days_since_last_update,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,engaged_sessions_90d,impressions_last_30d,clicks_last_30d,sessions_last_30d,...,impression_tier_good,impression_tier_low,impression_tier_moderate,impression_tier_nan,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,position_tier_nan
0,187,20,3803,29,22,17,1,578,2,2,...,1,0,0,0,0,0,0,1,0,0
1,445,25,15320,7,10,9,0,2501,2,3,...,1,0,0,0,0,0,1,0,0,0
2,141,20,12581,11,14,11,0,2382,1,1,...,1,0,0,0,0,0,1,0,0,0
3,463,22,11751,58,87,78,1,3626,22,35,...,1,0,0,0,0,1,0,0,0,0
4,263,14,19140,24,177,145,0,4211,10,14,...,1,0,0,0,0,0,1,0,0,0


### Feature vector

I build the feature vector from observable content and search-performance signals that are available before a page is selected for review.

The selected features describe content age, freshness, search visibility, engagement, and content size. Categorical fields are converted to numeric indicators, and missing numeric values are filled with the median of the available data.

I exclude identifiers such as `content_id` and `client_id` from the model features because they identify records or clients rather than representing generalizable content signals.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [12]:
# Feature audit table

feature_notes = []

for col in numeric_features:
    feature_notes.append({
        "feature": col,
        "type": "numeric",
        "missing_values": int(df[col].isna().sum()),
        "missing_handling": "median fill",
        "used_as_model_feature": True
    })

for col in categorical_features:
    feature_notes.append({
        "feature": col,
        "type": "categorical",
        "missing_values": int(df[col].isna().sum()),
        "missing_handling": "separate missing category",
        "used_as_model_feature": True
    })

feature_notes_df = pd.DataFrame(feature_notes)

display(feature_notes_df)

,feature,type,missing_values,missing_handling,used_as_model_feature
0,content_age_days,numeric,0,median fill,True
1,days_since_last_update,numeric,0,median fill,True
2,impressions_90d,numeric,0,median fill,True
3,clicks_90d,numeric,0,median fill,True
4,pageviews_90d,numeric,0,median fill,True
5,sessions_90d,numeric,0,median fill,True
6,engaged_sessions_90d,numeric,0,median fill,True
7,impressions_last_30d,numeric,0,median fill,True
8,clicks_last_30d,numeric,0,median fill,True
9,sessions_last_30d,numeric,0,median fill,True


## Feature notes

The main feature groups have the following meanings:

| Feature group | Meaning | Missing values | Available before prediction? |
|---|---|---|---|
| Content age | How old the content is | Median fill | Yes |
| Days since last update | Time since the content was updated | Median fill | Yes |
| Impressions | Search visibility during the observed window | Median fill | Yes |
| Clicks | Search clicks during the observed window | Median fill | Yes |
| Sessions / pageviews | Observed traffic and usage | Median fill | Yes |
| CTR | Click-through rate | Median fill | Yes |
| Average position | Average observed search position | Median fill | Yes |
| Engagement rate | Observed engagement signal | Median fill | Yes |
| Scroll rate | Observed scrolling behavior | Median fill | Yes |
| Word count / character count | Content size | Median fill | Yes |
| Content type / intent | Content characteristics | Categorical missing category | Yes |
| Freshness / age tiers | Derived content-age categories | Categorical missing category | Yes |

The features are intended to represent information observable during the measurement window before a human uses the model to prioritize pages for review.

The feature vector does not use `content_id` or `client_id` as predictive variables.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [13]:
# Potential leakage / identifier fields

leakage_keywords = [
    "declin",
    "label",
    "target",
    "future",
    "outcome",
    "next",
    "prediction"
]

identifier_keywords = [
    "id",
    "client"
]

potential_leakage = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

potential_identifiers = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in identifier_keywords)
]

print("Potential label/future leakage fields:")
print(potential_leakage)

print("\nPotential identifier fields:")
print(potential_identifiers)

# Confirm that the feature matrix does not contain identifiers
feature_identifier_overlap = [
    col for col in X.columns
    if col in ["content_id", "client_id"]
]

print("\nIdentifiers present in final feature matrix:")
print(feature_identifier_overlap)

assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("\nLeakage check passed: content_id and client_id are not model features.")

Potential label/future leakage fields:
[]

Potential identifier fields:
['content_id', 'client_id', 'provider_used']

Identifiers present in final feature matrix:
[]

Leakage check passed: content_id and client_id are not model features.


## Leakage hunt

I checked the feature set for three main forms of leakage:

1. Label-derived information — features that directly contain or reveal the target.
2. Future-window information — measurements that would only become available after the prediction point.
3. Identifier/product information — fields that identify a page, client, provider, or model rather than representing a general content signal.

The target label is not included in the feature matrix.

I also exclude identifiers and fields that could make the model depend on a specific client or record rather than on observable content/search characteristics.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [14]:
# Final feature summary

print("Final feature count:", X.shape[1])
print("Rows:", X.shape[0])

print("\nIdentifiers excluded:")
for col in ["content_id", "client_id"]:
    print(f"- {col}: {'excluded' if col not in X.columns else 'WARNING: included'}")

print("\nTarget leakage check:")
target_like = [
    col for col in X.columns
    if any(k in col.lower() for k in ["label", "target", "declin", "future", "outcome"])
]

print("Target-like columns in X:", target_like)

assert len(target_like) == 0

print("Leakage audit completed successfully.")

Final feature count: 59
Rows: 30000

Identifiers excluded:
- content_id: excluded
- client_id: excluded

Target leakage check:
Target-like columns in X: []
Leakage audit completed successfully.


## What I excluded and why

The following fields were excluded from the feature vector:

| Field / type | Reason for exclusion |
|---|---|
| `content_id` | Record identifier; it does not represent a generalizable content signal. |
| `client_id` | Client identifier; using it as a feature could allow the model to learn client-specific patterns. |
| Provider/model identifiers | These describe how data was produced rather than the content opportunity itself. |
| Target/decline label | This is the outcome being predicted and must not be used as an input feature. |
| Future outcome fields | These would reveal information that would not be available when making a review decision. |
| Private query/domain information | Excluded to maintain the public-safe nature of the analysis. |

The final feature set therefore focuses on observable content, freshness, search-performance, and engagement signals.

This audit does not prove that every possible leakage pathway has been eliminated. It documents the checks performed on the available fields.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.